In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os, math, random, gc, time
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ==================== SECTION 1: CONFIGURATION ====================
DATA_PATH_OPTIONS = [
    "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2"
]
DATA_PATH = None
for p in DATA_PATH_OPTIONS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    inp = "/kaggle/input"
    if os.path.exists(inp):
        for d in os.listdir(inp):
            cand = os.path.join(inp, d)
            if os.path.isdir(cand):
                if os.path.exists(os.path.join(cand, "test_in")):
                    DATA_PATH = cand; break
                if os.path.exists(os.path.join(cand, "raw")):
                    DATA_PATH = cand; break
                for sub in os.listdir(cand):
                    sc = os.path.join(cand, sub)
                    if os.path.isdir(sc) and (os.path.exists(os.path.join(sc, "test_in"))
                                              or os.path.exists(os.path.join(sc, "raw"))):
                        DATA_PATH = sc; break
                if DATA_PATH: break
assert DATA_PATH is not None, "Cannot find competition data!"
print(f"DATA_PATH: {DATA_PATH}")

# Constants
WINDOW = 26
INPUT_STEPS = 10
OUTPUT_STEPS = 16
VAL_HOURS = 72
ALL_MONTHS = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]

# Feature groups
TARGET = "cpm25"
MET_FEATURES = ["q2", "t2", "u10", "v10", "swdown", "pblh", "psfc", "rain"]
EMISSION_FEATURES = ["PM25", "NH3", "SO2", "NOx", "NMVOC_e", "NMVOC_finn", "bio"]
DERIVED_FEATURES = ["wind_speed", "wind_dir"]
ALL_RAW_FEATURES = [TARGET] + MET_FEATURES + EMISSION_FEATURES

# Feature order per timestep (channel 0 MUST be cpm25 for skip connection)
FEATURE_ORDER = [TARGET] + MET_FEATURES + EMISSION_FEATURES + DERIVED_FEATURES
N_FEAT = len(FEATURE_ORDER)  # 18
print(f"Features per timestep: {N_FEAT} = {FEATURE_ORDER}")

# Hyperparameters
ENC_DIM = 64
GRU_DIM = 64
FNO_WIDTH = 48
FNO_MODES = (20, 16)
FNO_BLOCKS = 4
DROPOUT = 0.1
BATCH_SIZE = 6
NUM_EPOCHS = 40
PATIENCE = 15
LR = 4e-4
WEIGHT_DECAY = 1e-2
NUM_ENSEMBLE = 3

# Loss weights
W_GLOBAL_SMAPE = 1.0
W_EP_SMAPE = 2.5
W_EP_CORR = 2.0
W_MSE = 0.05
SMAPE_EPS = 0.01  # NOT 1.0! Matches competition formula


# ==================== SECTION 2: PREPROCESSOR ====================
class Preprocessor:
    def __init__(self):
        self.stats = {}         # global per-feature stats
        self.gw_mean = None     # grid-wise PM2.5 mean (140, 124)
        self.gw_std = None      # grid-wise PM2.5 std (140, 124)

    def compute_stats(self, months):
        base = os.path.join(DATA_PATH, "raw")

        # --- Met features: global z-score with [1,99] percentile clipping ---
        for feat in MET_FEATURES:
            arrs = [np.load(os.path.join(base, m, f"{feat}.npy")).astype(np.float32) for m in months]
            combined = np.concatenate(arrs, axis=0).ravel()
            valid = combined[~np.isnan(combined)]
            p1, p99 = np.percentile(valid, [1, 99])
            c = np.clip(valid, p1, p99)
            self.stats[feat] = {"mean": float(np.mean(c)), "std": float(np.std(c)) + 1e-6}
            print(f"  MET  {feat:10s}: μ={self.stats[feat]['mean']:.4f}, σ={self.stats[feat]['std']:.4f}")

        # --- Emission features: global z-score WITHOUT log1p, WITHOUT percentile clip ---
        for feat in EMISSION_FEATURES:
            arrs = []
            for m in months:
                fpath = os.path.join(base, m, f"{feat}.npy")
                if os.path.exists(fpath):
                    arrs.append(np.load(fpath).astype(np.float32))
            if not arrs:
                self.stats[feat] = {"mean": 0., "std": 1.}
                print(f"  EMIS {feat:10s}: NOT FOUND, using zeros")
                continue
            combined = np.concatenate(arrs, axis=0).ravel()
            valid = combined[~np.isnan(combined)]
            std_val = float(np.std(valid))
            if std_val < 1e-10:
                self.stats[feat] = {"mean": 0., "std": 1.}
                print(f"  EMIS {feat:10s}: μ=0, σ~0 (constant, fallback std=1)")
            else:
                self.stats[feat] = {"mean": float(np.mean(valid)), "std": std_val + 1e-6}
                print(f"  EMIS {feat:10s}: μ={self.stats[feat]['mean']:.6f}, σ={self.stats[feat]['std']:.6f}")

        # --- Derived wind features ---
        u = np.concatenate([np.load(os.path.join(base, m, "u10.npy")).astype(np.float32) for m in months])
        v = np.concatenate([np.load(os.path.join(base, m, "v10.npy")).astype(np.float32) for m in months])
        ws = np.sqrt(u**2 + v**2)
        self.stats["wind_speed"] = {"mean": float(np.mean(ws)), "std": float(np.std(ws)) + 1e-6}
        self.stats["wind_dir"] = {"mean": 0.0, "std": float(np.pi)}
        print(f"  DER  wind_speed: μ={self.stats['wind_speed']['mean']:.4f}, σ={self.stats['wind_speed']['std']:.4f}")
        print(f"  DER  wind_dir  : μ=0.0, σ=π")
        del u, v, ws

        # --- PM2.5: grid-wise normalization ---
        pm = np.concatenate([np.load(os.path.join(base, m, "cpm25.npy")).astype(np.float32) for m in months])
        self.gw_mean = np.mean(pm, axis=0)   # (140, 124)
        self.gw_std = np.std(pm, axis=0) + 1e-6
        global_mean = float(np.mean(pm))
        global_std = float(np.std(pm)) + 1e-6
        # Fallback for low-variance grid points
        low = self.gw_std < 1.0
        self.gw_mean[low] = global_mean
        self.gw_std[low] = global_std
        print(f"  TGT  cpm25 (gridwise): mean∈[{self.gw_mean.min():.1f},{self.gw_mean.max():.1f}], "
              f"std∈[{self.gw_std.min():.1f},{self.gw_std.max():.1f}], fallback={low.sum()}")
        del pm

print("\n--- Computing Normalization Statistics ---")
prep = Preprocessor()
prep.compute_stats(ALL_MONTHS)

# Move grid-wise stats to GPU
GW_MEAN_NP = prep.gw_mean.copy()
GW_STD_NP = prep.gw_std.copy()
GW_MEAN = torch.from_numpy(GW_MEAN_NP).float().to(device)  # (140, 124)
GW_STD = torch.from_numpy(GW_STD_NP).float().to(device)


# ==================== SECTION 3: EPISODE MASKS (PARALLEL STL) ====================
print("\n--- Computing Episode Masks (Parallel STL) ---")
t0_ep = time.time()

from statsmodels.tsa.seasonal import STL
from joblib import Parallel, delayed

def _stl_one_gridpoint(ts, period=24):
    """Compute episode mask for a single grid point time series."""
    if np.std(ts) < 1e-6:
        return np.zeros(len(ts), dtype=np.float32)
    try:
        r = STL(ts, period=period, robust=True).fit().resid
        s = np.std(r) + 1e-8
        return ((r > 3 * s) & (ts > 1)).astype(np.float32)
    except Exception:
        return np.zeros(len(ts), dtype=np.float32)

ep_masks = []
for m in ALL_MONTHS:
    pm = np.load(os.path.join(DATA_PATH, "raw", m, "cpm25.npy")).astype(np.float32)
    T, H, W = pm.shape
    d2 = pm.reshape(T, -1)
    masks = Parallel(n_jobs=-1, backend='loky', verbose=0)(
        delayed(_stl_one_gridpoint)(d2[:, k]) for k in range(d2.shape[1]))
    mask = np.stack(masks, axis=1).reshape(T, H, W)
    ep_pct = 100 * mask.mean()
    print(f"  {m}: {T}h, {ep_pct:.2f}% episodic")
    ep_masks.append(mask)
    del pm

episode_mask_concat = np.concatenate(ep_masks, axis=0)
print(f"  Total: {episode_mask_concat.shape}, {100*episode_mask_concat.mean():.2f}% episodic")
print(f"  Episode mask time: {time.time()-t0_ep:.0f}s")


# ==================== SECTION 4: DATASET ====================
class PMDataset(Dataset):
    def __init__(self, months, split="train", stride=1):
        self.is_train = (split == "train")
        base = os.path.join(DATA_PATH, "raw")

        # Cache all raw features concatenated across months
        self.raw = {}
        for feat in ALL_RAW_FEATURES:
            arrs = []
            for m in months:
                fpath = os.path.join(base, m, f"{feat}.npy")
                if os.path.exists(fpath):
                    arrs.append(np.load(fpath).astype(np.float32))
                else:
                    ref_shape = np.load(os.path.join(base, m, "cpm25.npy")).shape
                    arrs.append(np.zeros(ref_shape, dtype=np.float32))
            self.raw[feat] = np.concatenate(arrs, axis=0)

        # Derived features
        self.raw["wind_speed"] = np.sqrt(self.raw["u10"]**2 + self.raw["v10"]**2)
        self.raw["wind_dir"] = np.arctan2(self.raw["v10"], self.raw["u10"])

        # Build sample indices
        self.indices = []
        offset = 0
        for m in months:
            mlen = np.load(os.path.join(base, m, "cpm25.npy")).shape[0]
            if split == "train":
                for i in range(0, mlen - VAL_HOURS - WINDOW + 1, stride):
                    self.indices.append(offset + i)
            else:
                for i in range(max(0, mlen - VAL_HOURS), mlen - WINDOW + 1, stride):
                    self.indices.append(offset + i)
            offset += mlen
        print(f"[{split}] {len(self.indices)} samples")

    def __len__(self):
        return len(self.indices)

    def _norm_feat(self, data, feat):
        s = prep.stats[feat]
        return (data - s["mean"]) / s["std"]

    def __getitem__(self, idx):
        s = self.indices[idx]

        # Build input: (T=10, C=18, H=140, W=124)
        x = np.zeros((INPUT_STEPS, N_FEAT, 140, 124), dtype=np.float32)
        for t in range(INPUT_STEPS):
            ch = 0
            # Channel 0: cpm25 — grid-wise normalization
            x[t, ch] = (self.raw[TARGET][s + t] - GW_MEAN_NP) / GW_STD_NP
            ch += 1
            # Channels 1-8: met — global z-score
            for feat in MET_FEATURES:
                x[t, ch] = self._norm_feat(self.raw[feat][s + t], feat)
                ch += 1
            # Channels 9-15: emissions — global z-score (no log1p)
            for feat in EMISSION_FEATURES:
                x[t, ch] = self._norm_feat(self.raw[feat][s + t], feat)
                ch += 1
            # Channels 16-17: derived wind
            x[t, ch] = self._norm_feat(self.raw["wind_speed"][s + t], "wind_speed")
            ch += 1
            x[t, ch] = self._norm_feat(self.raw["wind_dir"][s + t], "wind_dir")
            ch += 1

        # Target: (16, 140, 124) — grid-wise normalized
        target = (self.raw[TARGET][s + INPUT_STEPS:s + WINDOW] - GW_MEAN_NP) / GW_STD_NP

        # Episode mask: (16, 140, 124)
        ep_mask = episode_mask_concat[s + INPUT_STEPS:s + WINDOW]

        x = torch.from_numpy(x)
        target = torch.from_numpy(target)
        ep_mask = torch.from_numpy(ep_mask.copy())

        # --- AUGMENTATION (NO FLIPS! Geographic data is not flip-invariant) ---
        if self.is_train:
            # Gaussian noise (40% probability)
            if random.random() > 0.6:
                x = x + torch.randn_like(x) * 0.02
            # Feature dropout: zero out 1 random non-PM2.5 channel (20% probability)
            if random.random() > 0.8:
                ch_drop = random.randint(1, N_FEAT - 1)  # Never drop channel 0 (PM2.5)
                x[:, ch_drop] = 0

        return x, target, ep_mask


print("\n--- Building Datasets ---")
train_ds = PMDataset(ALL_MONTHS, "train")
val_ds = PMDataset(ALL_MONTHS, "val")
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True, persistent_workers=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=2, pin_memory=True, persistent_workers=True)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")


# ==================== SECTION 5: MODEL ====================

class ConvGRUCell(nn.Module):
    def __init__(self, in_ch, hid_ch, kernel=3):
        super().__init__()
        self.hid_ch = hid_ch
        p = kernel // 2
        self.gates = nn.Conv2d(in_ch + hid_ch, 2 * hid_ch, kernel, padding=p)
        self.cand = nn.Conv2d(in_ch + hid_ch, hid_ch, kernel, padding=p)

    def forward(self, x, h):
        if h is None:
            h = torch.zeros(x.shape[0], self.hid_ch, x.shape[2], x.shape[3], device=x.device)
        cat = torch.cat([x, h], 1)
        rz = torch.sigmoid(self.gates(cat))
        r, z = rz.chunk(2, dim=1)
        return z * h + (1 - z) * torch.tanh(self.cand(torch.cat([x, r * h], 1)))


class SpectralConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1, self.modes2 = modes1, modes2
        scale = math.sqrt(2.0 / (in_ch + out_ch))
        self.w1 = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes1, modes2, 2))
        self.w2 = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes1, modes2, 2))

    def _cmul(self, x, w):
        return torch.einsum("bixy,ioxy->boxy", x, torch.view_as_complex(w.contiguous()))

    def forward(self, x):
        B, C, H, W = x.shape
        xft = torch.fft.rfft2(x.float())  # Ensure float32 for FFT
        m1 = min(self.modes1, H)
        m2 = min(self.modes2, W // 2 + 1)
        out = torch.zeros(B, self.w1.shape[1], H, W // 2 + 1,
                          dtype=torch.cfloat, device=x.device)
        out[:, :, :m1, :m2] = self._cmul(xft[:, :, :m1, :m2], self.w1[:, :, :m1, :m2])
        out[:, :, -m1:, :m2] = self._cmul(xft[:, :, -m1:, :m2], self.w2[:, :, :m1, :m2])
        return torch.fft.irfft2(out, s=(H, W))


class FNO2DBlock(nn.Module):
    def __init__(self, ch, modes1, modes2, drop=0.1):
        super().__init__()
        self.spec = SpectralConv2d(ch, ch, modes1, modes2)
        self.local_conv = nn.Conv2d(ch, ch, 3, padding=1)
        self.norm = nn.GroupNorm(8, ch)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        h = self.spec(x) + self.local_conv(x)
        h = self.norm(h)
        return x + self.drop(F.gelu(h))


class HybridFNOGRU(nn.Module):
    """
    Hybrid ConvGRU + FNO2D with Direct Prediction.
    - ConvGRU: temporal encoding over 10 input steps
    - FNO2D: long-range spatial processing via spectral convolutions
    - Direct head: predict all 16 output steps at once (no AR rollout)
    - Persistence skip: add last PM2.5 × learnable gain
    """
    def __init__(self, n_feat=N_FEAT, enc_dim=ENC_DIM, gru_dim=GRU_DIM,
                 fno_width=FNO_WIDTH, fno_modes=FNO_MODES,
                 n_fno_blocks=FNO_BLOCKS, output_steps=OUTPUT_STEPS, drop=DROPOUT):
        super().__init__()

        # Per-timestep feature encoder
        self.feat_enc = nn.Sequential(
            nn.Conv2d(n_feat, enc_dim, 1),
            nn.GELU(),
            nn.Conv2d(enc_dim, enc_dim, 3, padding=1),
            nn.GELU(),
            nn.GroupNorm(8, enc_dim),
        )

        # Temporal encoder (ConvGRU)
        self.gru = ConvGRUCell(enc_dim, gru_dim, kernel=3)

        # Lift GRU hidden state to FNO width
        self.lift = nn.Sequential(
            nn.Conv2d(gru_dim, fno_width, 1),
            nn.GELU(),
        )

        # FNO2D spatial processing blocks
        self.fno_blocks = nn.ModuleList([
            FNO2DBlock(fno_width, fno_modes[0], fno_modes[1], drop)
            for _ in range(n_fno_blocks)
        ])

        # Prediction head: all 16 output steps at once
        self.head = nn.Sequential(
            nn.Conv2d(fno_width, fno_width, 3, padding=1),
            nn.GELU(),
            nn.GroupNorm(8, fno_width),
            nn.Conv2d(fno_width, output_steps, 1),
        )

        # Persistence skip: last PM2.5 × learnable per-step gain
        self.skip_gain = nn.Parameter(torch.ones(output_steps))

    def forward(self, x):
        """
        x: (B, T=10, C=18, H=140, W=124)
        returns: (B, 16, 140, 124) — normalized PM2.5 predictions
        """
        B, T, C, H, W = x.shape

        # Step 1: Temporal encoding via ConvGRU
        h = None
        for t in range(T):
            feat = self.feat_enc(x[:, t])   # (B, enc_dim, H, W)
            h = self.gru(feat, h)           # (B, gru_dim, H, W)

        # Step 2: Lift to FNO width
        h = self.lift(h)                    # (B, fno_width, H, W)

        # Step 3: FNO2D spatial processing
        for block in self.fno_blocks:
            h = block(h)                    # (B, fno_width, H, W)

        # Step 4: Predict all 16 output steps
        out = self.head(h)                  # (B, 16, H, W)

        # Step 5: Persistence skip connection
        last_pm = x[:, -1, 0:1, :, :]      # (B, 1, H, W) — channel 0 = cpm25
        out = out + last_pm * self.skip_gain.view(1, -1, 1, 1)

        return out


# ==================== SECTION 6: LOSS FUNCTION ====================

def metric_aligned_loss(pred, target, ep_mask):
    """
    Loss aligned with the EXACT Phase 2 competition metric:
      Score = w1*NormGlobalSMAPE + w2*NormEpisodeCorr + w3*NormEpisodeSMAPE

    All SMAPE computations use the competition formula with ε=0.01 (NOT +1.0).
    Episode correlation is computed per-timestep then averaged (matches competition).
    """
    # Denormalize to raw PM2.5 scale (µg/m³)
    pred_raw = torch.clamp(pred * GW_STD[None, None] + GW_MEAN[None, None], min=0.0)
    tgt_raw = torch.clamp(target * GW_STD[None, None] + GW_MEAN[None, None], min=0.0)

    # --- 1. Global SMAPE (exact competition formula) ---
    denom = 0.5 * (pred_raw + tgt_raw) + SMAPE_EPS
    smape_pointwise = (pred_raw - tgt_raw).abs() / denom
    global_smape = smape_pointwise.mean()

    # --- 2. Episode SMAPE ---
    ep_count = ep_mask.sum()
    if ep_count > 0:
        ep_smape = (smape_pointwise * ep_mask).sum() / ep_count
    else:
        ep_smape = torch.tensor(0.0, device=pred.device)

    # --- 3. Episode Correlation (PER TIMESTEP — exact competition metric) ---
    B, T, H, W = pred_raw.shape
    corr_losses = []
    for t in range(T):
        mask_t = ep_mask[:, t]              # (B, H, W)
        p_ep = pred_raw[:, t][mask_t.bool()]  # (N_episodic,)
        g_ep = tgt_raw[:, t][mask_t.bool()]
        if p_ep.numel() > 10:
            p_c = p_ep - p_ep.mean()
            g_c = g_ep - g_ep.mean()
            p_norm = p_c.norm()
            g_norm = g_c.norm()
            if p_norm > 1e-8 and g_norm > 1e-8:
                corr = (p_c * g_c).sum() / (p_norm * g_norm)
                corr_losses.append(1.0 - corr)
    if corr_losses:
        ep_corr_loss = torch.stack(corr_losses).mean()
    else:
        ep_corr_loss = torch.tensor(0.0, device=pred.device)

    # --- 4. MSE anchor (small weight for gradient stability) ---
    mse = ((pred - target) ** 2).mean()

    # --- Combined loss ---
    loss = (W_GLOBAL_SMAPE * global_smape +
            W_EP_SMAPE * ep_smape +
            W_EP_CORR * ep_corr_loss +
            W_MSE * mse)

    return loss


def mixup_data(x, tgt, mask, alpha=0.2):
    """Mixup augmentation (safe for geographic data)."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return (lam * x + (1 - lam) * x[idx],
            lam * tgt + (1 - lam) * tgt[idx],
            torch.maximum(mask, mask[idx]))


# ==================== SECTION 7: TRAINING ====================

def train_epoch(model, loader, opt, scaler):
    model.train()
    total_loss, n_batches = 0.0, 0
    for x, tgt, mask in loader:
        x, tgt, mask = x.to(device), tgt.to(device), mask.to(device)

        # Mixup (50% probability)
        if random.random() > 0.5:
            x, tgt, mask = mixup_data(x, tgt, mask)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            pred = model(x)
            loss = metric_aligned_loss(pred, tgt, mask)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


def validate_metrics(model, loader):
    """Compute exact competition metrics on validation set."""
    model.eval()
    all_pred, all_tgt, all_mask = [], [], []

    with torch.no_grad(), torch.amp.autocast('cuda'):
        for x, tgt, mask in loader:
            x = x.to(device)
            pred = model(x).float()
            all_pred.append(pred.cpu())
            all_tgt.append(tgt)
            all_mask.append(mask)

    pred = torch.cat(all_pred)      # (N, 16, 140, 124)
    tgt = torch.cat(all_tgt)
    mask = torch.cat(all_mask)

    # Denormalize
    gw_mean = torch.from_numpy(GW_MEAN_NP).float()
    gw_std = torch.from_numpy(GW_STD_NP).float()
    pred_raw = torch.clamp(pred * gw_std[None, None] + gw_mean[None, None], min=0)
    tgt_raw = torch.clamp(tgt * gw_std[None, None] + gw_mean[None, None], min=0)

    # 1. Global SMAPE
    denom = 0.5 * (pred_raw + tgt_raw) + SMAPE_EPS
    smape = (pred_raw - tgt_raw).abs() / denom
    global_smape = smape.mean().item()

    # 2. Episode SMAPE
    ep_n = mask.sum().item()
    ep_smape = (smape * mask).sum().item() / (ep_n + 1e-8) if ep_n > 0 else 0.0

    # 3. Episode Correlation (per-timestep)
    corrs = []
    for t in range(OUTPUT_STEPS):
        mask_t = mask[:, t]
        p = pred_raw[:, t][mask_t.bool()]
        g = tgt_raw[:, t][mask_t.bool()]
        if p.numel() > 1:
            p_c = p - p.mean()
            g_c = g - g.mean()
            pn, gn = p_c.norm(), g_c.norm()
            if pn > 1e-8 and gn > 1e-8:
                corrs.append(((p_c * g_c).sum() / (pn * gn)).item())
    ep_corr = np.mean(corrs) if corrs else 0.0

    # Normalized scores (competition formula)
    norm_corr = (ep_corr + 1) / 2
    norm_global = 1 - global_smape / 2
    norm_ep_smape = 1 - ep_smape / 2

    # RMSE (for comparison with previous models)
    rmse = math.sqrt(((pred_raw - tgt_raw) ** 2).mean().item())

    return {
        "rmse": rmse,
        "global_smape": global_smape,
        "ep_smape": ep_smape,
        "ep_corr": ep_corr,
        "norm_corr": norm_corr,
        "norm_global": norm_global,
        "norm_ep_smape": norm_ep_smape,
    }


# ==================== SANITY CHECK ====================
print("\n--- Sanity Check ---")
model_test = HybridFNOGRU().to(device)
x_t, tgt_t, mask_t = next(iter(train_dl))
x_t, tgt_t, mask_t = x_t.to(device), tgt_t.to(device), mask_t.to(device)
print(f"Input:  {x_t.shape}  — expected (B, 10, 18, 140, 124)")
print(f"Target: {tgt_t.shape} — expected (B, 16, 140, 124)")
print(f"Mask:   {mask_t.shape} — expected (B, 16, 140, 124)")

with torch.amp.autocast('cuda'):
    out_t = model_test(x_t)
    loss_t = metric_aligned_loss(out_t, tgt_t, mask_t)

print(f"Output: {out_t.shape}  — expected (B, 16, 140, 124)")
print(f"Loss:   {loss_t.item():.4f}")
n_params = sum(p.numel() for p in model_test.parameters())
print(f"Params: {n_params:,}")
print(f"Skip gains: [{', '.join(f'{g:.2f}' for g in model_test.skip_gain.detach().cpu().numpy())}]")
assert out_t.shape == tgt_t.shape, f"Shape mismatch: {out_t.shape} vs {tgt_t.shape}"
assert not torch.isnan(out_t).any(), "NaN in output!"
assert not torch.isnan(loss_t), "NaN loss!"
del model_test, out_t, loss_t, x_t, tgt_t, mask_t
torch.cuda.empty_cache()
print("✅ Sanity check passed\n")


# ==================== ENSEMBLE TRAINING ====================
print("=" * 60)
print("TRAINING: 3-model ensemble | HybridFNOGRU | Metric-aligned loss")
print("=" * 60)

saved_paths = []
best_vals = []
t0_total = time.time()

for ei in range(NUM_ENSEMBLE):
    seed = 42 + ei * 97
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = HybridFNOGRU().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, steps_per_epoch=len(train_dl),
        epochs=NUM_EPOCHS, pct_start=0.15, anneal_strategy='cos')
    scaler = torch.amp.GradScaler('cuda')

    best_metric = float('inf')
    best_ep = 0
    path = f"/kaggle/working/hybrid_fno_gru_{ei}.pt"

    print(f"\n{'='*50}")
    print(f"ENSEMBLE {ei+1}/{NUM_ENSEMBLE} (seed={seed})")
    print(f"{'='*50}")
    t0 = time.time()

    for epoch in range(NUM_EPOCHS):
        tl = train_epoch(model, train_dl, opt, scaler)
        sched.step()

        # Validate every 3 epochs or at end
        if epoch % 3 == 0 or epoch == NUM_EPOCHS - 1 or epoch >= NUM_EPOCHS - 5:
            metrics = validate_metrics(model, val_dl)
            # Use negative of estimated score as metric (lower = better for early stopping)
            val_metric = metrics["global_smape"] + metrics["ep_smape"] - metrics["ep_corr"]
            elapsed = (time.time() - t0) / 60

            print(f"  Ep {epoch+1:3d} | TrLoss {tl:.4f} | "
                  f"RMSE {metrics['rmse']:.1f} | "
                  f"GSmape {metrics['global_smape']:.4f} | "
                  f"EpSmape {metrics['ep_smape']:.4f} | "
                  f"EpCorr {metrics['ep_corr']:.4f} | "
                  f"NormG {metrics['norm_global']:.4f} | "
                  f"NormC {metrics['norm_corr']:.4f} | "
                  f"NormES {metrics['norm_ep_smape']:.4f} | "
                  f"{elapsed:.1f}m")

            if val_metric < best_metric:
                best_metric = val_metric
                best_ep = epoch
                torch.save(model.state_dict(), path)
                print(f"           *** Best (val_metric={val_metric:.4f}) ***")

            if epoch - best_ep >= PATIENCE:
                print(f"  Early stop at epoch {epoch+1} ({elapsed:.1f}m)")
                break
        else:
            # Light validation (just train loss print)
            if epoch % 1 == 0:
                elapsed = (time.time() - t0) / 60
                print(f"  Ep {epoch+1:3d} | TrLoss {tl:.4f} | {elapsed:.1f}m")

    # Final metrics for this model
    model.load_state_dict(torch.load(path, weights_only=True))
    final_metrics = validate_metrics(model, val_dl)
    print(f"  Final: RMSE={final_metrics['rmse']:.1f}, EpCorr={final_metrics['ep_corr']:.4f}, "
          f"GSmape={final_metrics['global_smape']:.4f}")
    print(f"  Skip gains: [{', '.join(f'{g:.2f}' for g in model.skip_gain.detach().cpu().numpy())}]")

    saved_paths.append(path)
    best_vals.append(best_metric)
    del model, opt, sched, scaler
    torch.cuda.empty_cache()
    gc.collect()

train_time = (time.time() - t0_total) / 60
print(f"\n*** Total training: {train_time:.1f} min ***")


# ==================== SECTION 8: INFERENCE ====================
print("\n" + "=" * 60)
print("INFERENCE — Weighted ensemble, NO TTA flips, NO smoothing")
print("=" * 60)

# Free training data
del train_ds, val_ds, train_dl, val_dl
gc.collect()
torch.cuda.empty_cache()
print("Training data freed")

# Ensemble weights (inverse validation metric — lower metric = better model = higher weight)
weights = np.array([1.0 / (v + 1e-8) for v in best_vals])
weights = weights / weights.sum()
print(f"Ensemble weights: {weights}")
print(f"Val metrics: {best_vals}")

# Load test data
test_path = os.path.join(DATA_PATH, "test_in")
td = {}
for feat in ALL_RAW_FEATURES:
    fpath = os.path.join(test_path, f"{feat}.npy")
    if os.path.exists(fpath):
        td[feat] = np.load(fpath, mmap_mode="r")
    else:
        print(f"WARNING: {feat} missing from test_in, using zeros")
        ref = np.load(os.path.join(test_path, "cpm25.npy"), mmap_mode="r")
        td[feat] = np.zeros_like(ref)

nsamples = td["cpm25"].shape[0]
print(f"Test samples: {nsamples}")
print(f"Expected output: ({nsamples}, 140, 124, {OUTPUT_STEPS})")

# Accumulator for weighted ensemble predictions
ens_preds = np.zeros((nsamples, 140, 124, OUTPUT_STEPS), dtype=np.float32)

for ei, path in enumerate(saved_paths):
    print(f"\nModel {ei+1}/{NUM_ENSEMBLE} (weight={weights[ei]:.3f})")
    model = HybridFNOGRU().to(device)
    model.load_state_dict(torch.load(path, weights_only=True))
    model.eval()

    bs = 4  # Inference batch size (no gradients → more VRAM available)
    for s in range(0, nsamples, bs):
        e = min(s + bs, nsamples)
        cb = e - s

        # Build input tensor: (cb, 10, 18, 140, 124)
        x_np = np.zeros((cb, INPUT_STEPS, N_FEAT, 140, 124), dtype=np.float32)

        for b, i in enumerate(range(s, e)):
            for t in range(INPUT_STEPS):
                ch = 0
                # Channel 0: cpm25 — grid-wise normalized
                x_np[b, t, ch] = (td[TARGET][i][t].astype(np.float32) - GW_MEAN_NP) / GW_STD_NP
                ch += 1
                # Channels 1-8: met
                for feat in MET_FEATURES:
                    x_np[b, t, ch] = (td[feat][i][t].astype(np.float32) - prep.stats[feat]["mean"]) / prep.stats[feat]["std"]
                    ch += 1
                # Channels 9-15: emissions
                for feat in EMISSION_FEATURES:
                    x_np[b, t, ch] = (td[feat][i][t].astype(np.float32) - prep.stats[feat]["mean"]) / prep.stats[feat]["std"]
                    ch += 1
                # Channels 16-17: derived wind
                u = td["u10"][i][t].astype(np.float32)
                v = td["v10"][i][t].astype(np.float32)
                ws = np.sqrt(u**2 + v**2)
                wd = np.arctan2(v, u)
                x_np[b, t, ch] = (ws - prep.stats["wind_speed"]["mean"]) / prep.stats["wind_speed"]["std"]
                ch += 1
                x_np[b, t, ch] = (wd - prep.stats["wind_dir"]["mean"]) / prep.stats["wind_dir"]["std"]
                ch += 1

        xt = torch.from_numpy(x_np).to(device)

        # Forward pass — NO TTA flips!
        with torch.no_grad(), torch.amp.autocast('cuda'):
            pred = model(xt).float()

        # Denormalize
        pred_np = pred.cpu().numpy()
        pred_np = pred_np * GW_STD_NP[None, None] + GW_MEAN_NP[None, None]
        pred_np = np.clip(pred_np, 0, 500)

        # Accumulate: transpose (B, T, H, W) → (B, H, W, T)
        ens_preds[s:e] += pred_np.transpose(0, 2, 3, 1) * weights[ei]

        if s % 100 == 0:
            print(f"  {s}/{nsamples}")

    del model
    torch.cuda.empty_cache()
    gc.collect()

# NO temporal smoothing — preserves episode signals!
ens_preds = np.clip(ens_preds, 0, 500)

# Save
np.save("/kaggle/working/preds.npy", ens_preds.astype(np.float32))
total_time = (time.time() - t0_total) / 60

print(f"\n{'='*60}")
print(f"✅ DONE!")
print(f"  Shape: {ens_preds.shape}")
print(f"  Range: [{ens_preds.min():.2f}, {ens_preds.max():.2f}]")
print(f"  Mean:  {ens_preds.mean():.2f}")
print(f"  Total time: {total_time:.1f} min ({total_time/60:.1f} hrs)")
print(f"{'='*60}")


Device: cuda
DATA_PATH: /kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2
Features per timestep: 18 = ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir']

--- Computing Normalization Statistics ---
  MET  q2        : μ=0.0115, σ=0.0070
  MET  t2        : μ=291.6109, σ=13.9322
  MET  u10       : μ=1.5440, σ=3.4679
  MET  v10       : μ=0.1306, σ=2.8897
  MET  swdown    : μ=221.3180, σ=308.1456
  MET  pblh      : μ=759.4549, σ=613.7186
  MET  psfc      : μ=87994.2661, σ=17628.4709
  MET  rain      : μ=0.0556, σ=0.2546
  EMIS PM25      : μ=0.000000, σ=0.000001
  EMIS NH3       : μ=0, σ~0 (constant, fallback std=1)
  EMIS SO2       : μ=0.000000, σ=0.000001
  EMIS NOx       : μ=0.000000, σ=0.000001
  EMIS NMVOC_e   : μ=0.000000, σ=0.000001
  EMIS NMVOC_finn: μ=0.000000, σ=0.000001
  EMIS bio       : μ=0.000000, σ=0.000001
  DER  wind_spe